# 🐾 BabelNet — Explorando os 8 Synsets de "cachorro"

A busca retornou **8 synsets** para a palavra "cachorro".  
Cada um representa um **significado diferente** no BabelNet.  
Vamos descobrir o que cada ID representa!

---

In [1]:
# ─────────────────────────────────────────────────────────────
# Configuração inicial — carregar API Key e importações
# ─────────────────────────────────────────────────────────────

import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('BABELNET_KEY')

print(f'✅ API Key carregada: {API_KEY[:8]}...')

✅ API Key carregada: 66ebdb15...


In [2]:
# ─────────────────────────────────────────────────────────────
# Lista com os 8 IDs retornados pela busca de 'cachorro'
# Vamos consultar cada um para descobrir seu significado
# ─────────────────────────────────────────────────────────────

# IDs retornados pelo Exemplo 1
ids_cachorro = [
    'bn:00015267n',
    'bn:00055491n',
    'bn:00065248n',
    'bn:00010741n',
    'bn:00024245n',
    'bn:06913112n',
    'bn:00065261n',
    'bn:21937745n',
]

def buscar_detalhes_synset(synset_id, idioma_alvo='PT'):
    """
    Busca os detalhes de um synset no BabelNet pelo seu ID.
    Retorna o JSON completo com definições, lemas e categorias.
    """
    url = 'https://babelnet.io/v9/getSynset'
    params = {
        'id'        : synset_id,
        'targetLang': idioma_alvo,
        'key'       : API_KEY
    }
    resposta = requests.get(url, params=params)
    if resposta.status_code == 200:
        return resposta.json()
    else:
        print(f'  Erro {resposta.status_code} ao consultar {synset_id}')
        return {}


def extrair_resumo(synset_id, dados):
    """
    Extrai as informações mais relevantes de um synset:
    definição principal, lemas em PT e EN, e categorias.
    """
    resumo = {'id': synset_id}

    # ── Definição principal (preferir PT, senão EN)
    glosses = dados.get('glosses', [])
    definicao = '(sem definição disponível)'
    for g in glosses:
        if g.get('language') == 'PT':
            definicao = g['gloss']
            break
    if definicao == '(sem definição disponível)':
        for g in glosses:
            if g.get('language') == 'EN':
                definicao = g['gloss']
                break
    resumo['definicao'] = definicao

    # ── Lemas em Português e Inglês
    senses = dados.get('senses', [])
    lemas_pt, lemas_en = [], []
    for sense in senses:
        props = sense.get('properties', {})
        lang  = props.get('language', '')
        lemma = props.get('fullLemma', '')
        if lang == 'PT' and lemma and lemma not in lemas_pt:
            lemas_pt.append(lemma)
        if lang == 'EN' and lemma and lemma not in lemas_en:
            lemas_en.append(lemma)
    resumo['lemas_pt'] = lemas_pt[:4]   # Até 4 lemas em PT
    resumo['lemas_en'] = lemas_en[:4]   # Até 4 lemas em EN

    # ── Categorias / domínios
    categorias = dados.get('categories', [])
    resumo['categorias'] = [c.get('category', '') for c in categorias[:3]]

    return resumo


# ── Consultar todos os 8 synsets e armazenar os resumos
print('🔍 Consultando os 8 synsets de "cachorro" na API BabelNet...\n')

resumos = []
for i, syn_id in enumerate(ids_cachorro):
    print(f'   Consultando {i+1}/8: {syn_id}', end='\r')
    dados  = buscar_detalhes_synset(syn_id, idioma_alvo='PT')
    resumo = extrair_resumo(syn_id, dados)
    resumos.append(resumo)

print('\n\n✅ Consulta concluída! Exibindo resultados:\n')
print('=' * 65)

for i, r in enumerate(resumos):
    print(f'\n📌 Synset {i+1}: {r["id"]}')
    print(f'   📖 Definição : {r["definicao"]}')
    print(f'   🇧🇷 Lemas PT  : {", ".join(r["lemas_pt"]) if r["lemas_pt"] else "—"}')
    print(f'   🇺🇸 Lemas EN  : {", ".join(r["lemas_en"]) if r["lemas_en"] else "—"}')
    if r['categorias']:
        print(f'   🏷️  Categorias: {", ".join(r["categorias"])}')
    print('-' * 65)

🔍 Consultando os 8 synsets de "cachorro" na API BabelNet...

   Consultando 8/8: bn:21937745n

✅ Consulta concluída! Exibindo resultados:


📌 Synset 1: bn:00015267n
   📖 Definição : Um membro do gênero Canis (provavelmente descende do lobo comum) que tem sido domesticada pelo homem desde os tempos pré-históricos; ocorre em muitas raças.
   🇧🇷 Lemas PT  : cachorro, Canis_familiaris, cão, Canis_lupus_familiaris
   🇺🇸 Lemas EN  : —
   🏷️  Categorias: Animais_domésticos, Cães, Mamíferos_descritos_em_1758
-----------------------------------------------------------------

📌 Synset 2: bn:00055491n
   📖 Definição : Em arquitectura, designa-se como cachorro ou mísula a um elemento exposto que suporta os beirais de um telhado ou qualquer outro corpo saliente de um edifício, ao mesmo tempo que pode ter carácter decorativo, como acontece na Domus Municipalis de Bragança.
   🇧🇷 Lemas PT  : cachorro_(arquitetura), Cachorro_(arquitetura), mísula, cachorrada
   🇺🇸 Lemas EN  : —
   🏷️  Categorias: Elem

In [3]:
# ─────────────────────────────────────────────────────────────
# TABELA RESUMO — visão consolidada dos 8 synsets
# Mostra de forma clara como uma palavra pode ter
# múltiplos significados em um grafo de conhecimento
# ─────────────────────────────────────────────────────────────

import pandas as pd

# Montar uma tabela com os dados coletados
linhas = []
for i, r in enumerate(resumos):
    linhas.append({
        '#'         : i + 1,
        'ID'        : r['id'],
        'Lemas PT'  : ', '.join(r['lemas_pt']) if r['lemas_pt'] else '—',
        'Lemas EN'  : ', '.join(r['lemas_en']) if r['lemas_en'] else '—',
        'Definição' : r['definicao'][:80] + '...' if len(r['definicao']) > 80 else r['definicao'],
    })

df = pd.DataFrame(linhas).set_index('#')

print('📊 Tabela Resumo — 8 Synsets de "cachorro" no BabelNet\n')
print(df.to_string())

print()
print('💡 Conclusão:')
print('   A palavra "cachorro" possui múltiplos significados no BabelNet.')
print('   Cada synset representa um conceito distinto —')
print('   é exatamente isso que torna os grafos de conhecimento')
print('   tão poderosos para o PLN!')

📊 Tabela Resumo — 8 Synsets de "cachorro" no BabelNet

             ID                                                            Lemas PT Lemas EN                                                                            Definição
#                                                                                                                                                                                
1  bn:00015267n             cachorro, Canis_familiaris, cão, Canis_lupus_familiaris        —  Um membro do gênero Canis (provavelmente descende do lobo comum) que tem sido do...
2  bn:00055491n  cachorro_(arquitetura), Cachorro_(arquitetura), mísula, cachorrada        —  Em arquitectura, designa-se como cachorro ou mísula a um elemento exposto que su...
3  bn:00065248n                                                            cachorro        —                                                               Um jovem inexperiente.
4  bn:00010741n                                     cad